# Stress Detection from Wearable Data with a PyTorch MLP

A small PyTorch MLP for binary stress classification from sensor data: data loaders, training loop, loss tracking, and spotting overfitting.

_Datasets are not included in this repository._

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import torch
from torch import nn

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import make_classification

SEED = 16
np.random.seed(SEED)
torch.manual_seed(SEED)

df =pd.read_csv('wearable_stress_data.csv')

In [ ]:
X = df.drop('is_high_stress', axis=1)
y = df['is_high_stress']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=SEED,
)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
X_train.shape, y_train.shape

((1600, 6), (1600,))

In [ ]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

y_train_tensor.shape

dataset = TensorDataset(X_train_tensor, y_train_tensor)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [ ]:
class BasicMLP(nn.Module):
    def __init__(self, input, hidden1, hidden2, output):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input, hidden1),
            nn.ReLU(),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Linear(hidden2, output),
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
X_train.shape[1]

6

In [ ]:
model = BasicMLP(X_train.shape[1],32, 64, 1)

In [ ]:
lossy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [ ]:
epochs = 30
train_losses = []
val_losses = []

for epoch in range(1, epochs + 1):
    model.train()
    total_train_loss = 0.0

    for batch_X, batch_y in dataloader:
        predictions = model(batch_X)
        loss = lossy(predictions, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(dataloader)
    train_losses.append(avg_train_loss)

    model.eval()
    with torch.no_grad():
        val_preds = model(X_test_tensor)
        val_loss = lossy(val_preds, y_test_tensor)
        val_losses.append(val_loss.item())

    if epoch == 1 or epoch % 5 == 0:
        print(f"Epoch {epoch}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss.item():.4f}")

Epoch 1/30 | Train Loss: 0.3462 | Val Loss: 0.2750
Epoch 5/30 | Train Loss: 0.2372 | Val Loss: 0.2614
Epoch 10/30 | Train Loss: 0.2266 | Val Loss: 0.2565
Epoch 15/30 | Train Loss: 0.2162 | Val Loss: 0.2604
Epoch 20/30 | Train Loss: 0.2118 | Val Loss: 0.2553
Epoch 25/30 | Train Loss: 0.1918 | Val Loss: 0.2959
Epoch 30/30 | Train Loss: 0.1823 | Val Loss: 0.3011


In [ ]:
model.eval()
with torch.no_grad():
    train_logits = model(X_train_tensor)
    train_final_preds = (torch.sigmoid(train_logits) > 0.5).float()
    train_acc = (train_final_preds == y_train_tensor).float().mean().item()

    val_logits = model(X_test_tensor)
    val_final_preds = (torch.sigmoid(val_logits) > 0.5).float()
    val_acc = (val_final_preds == y_test_tensor).float().mean().item()

print(f"Final Training Accuracy: {train_acc:.2%}")
print(f"Final Validation Accuracy: {val_acc:.2%}")

Final Training Accuracy: 93.06%
Final Validation Accuracy: 87.00%


Model mengalami overfitting karena val loss tidak stabil selama epoch berlanjut meskipun train loss menurun terus.